# Cleaning MLK Speech & Sermon Transcripts

This notebook cleans raw Whisper transcript outputs so they are easier to use for NLP analysis.

It will take raw .txt transcript files from:

```text
/content/raw_transcripts
```

and create:

```text
/content/cleaned_transcripts
```

## Specific steps

- Loads raw transcript `.txt` files
- Normalizes spacing and line breaks
- Removes common transcription artifacts
- Optionally lowercases text for analysis
- Splits text into sentences
- Saves cleaned transcript files

## Import necessary libraries

In [ ]:
import re
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

try:
    import spacy
    nlp = spacy.load("en_core_web_sm")
except Exception as e:
    print("spaCy model not loaded. Run: !python -m spacy download en_core_web_sm")
    raise e

## Set input and output folders

In [ ]:
# Removing any files from local disk
!rm -rf /content/drive

# Make sure Google Drive is mounted first:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = Path("/content/drive/MyDrive/mlk_rhetoric_analysis")

# Folder containing raw Whisper transcript .txt files
RAW_TRANSCRIPT_DIR = BASE_DIR / "data" / "raw_transcripts"

# Folder where cleaned transcripts will be saved
CLEANED_TRANSCRIPT_DIR = BASE_DIR / "data" / "cleaned_transcripts"
CLEANED_TRANSCRIPT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", BASE_DIR)
print("Raw transcript folder:", RAW_TRANSCRIPT_DIR)
print("Cleaned transcript folder:", CLEANED_TRANSCRIPT_DIR)

Mounted at /content/drive
Project root: /content/drive/MyDrive/mlk_rhetoric_analysis
Raw transcript folder: /content/drive/MyDrive/mlk_rhetoric_analysis/data/raw_transcripts
Cleaned transcript folder: /content/drive/MyDrive/mlk_rhetoric_analysis/data/cleaned_transcripts


## Define cleaning functions

For this project, considering Martin Luther King, Jr.'s distinct sermonic delivery, rhetorical repetition matters, so the goal is to clean technical noise without erasing performance style.

In [ ]:
def normalize_whitespace(text: str) -> str:
    """Normalize spacing, tabs, and line breaks."""
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def remove_common_artifacts(text: str) -> str:
    """Remove common transcript artifacts that are usually not meaningful speech."""
    artifact_patterns = [
        r"\[music\]",
        r"\(music\)",
        r"♪",
    ]

    for pattern in artifact_patterns:
        text = re.sub(pattern, " ", text, flags=re.IGNORECASE)

    return normalize_whitespace(text)


def normalize_punctuation(text: str) -> str:
    """Clean up repeated or awkward punctuation spacing."""
    text = re.sub(r"\s+([,.!?;:])", r"\1", text)
    text = re.sub(r"([,.!?;:])([^\s\"'])", r"\1 \2", text)
    text = re.sub(r"([!?]){2,}", r"\1", text)
    text = re.sub(r"\.{3,}", "...", text)
    return normalize_whitespace(text)


def sentence_segment(text: str) -> list[str]:
    """Split transcript into sentences using spaCy."""
    doc = nlp(text)
    return [sent.text.strip() for sent in doc.sents if sent.text.strip()]


def clean_transcript(text: str, lowercase_for_analysis: bool = False) -> dict:
    """Clean transcript text and return multiple useful versions."""
    raw_text = text

    text = normalize_whitespace(text)
    text = remove_common_artifacts(text)
    text = normalize_punctuation(text)

    display_text = text
    analysis_text = text.lower() if lowercase_for_analysis else text

    sentences = sentence_segment(display_text)
    sentence_text = "\n".join(sentences)

    return {
        "raw_text": raw_text,
        "display_text": display_text,
        "analysis_text": analysis_text,
        "sentences": sentences,
        "sentence_text": sentence_text,
    }

## Preview one transcript

Verifying the cleaning behavior before processing the whole folder.

In [ ]:
txt_files = sorted(RAW_TRANSCRIPT_DIR.glob("*.txt"))

print(f"Found {len(txt_files)} transcript files")

if txt_files:
    sample_file = txt_files[0]
    raw_sample = sample_file.read_text(encoding="utf-8")
    cleaned_sample = clean_transcript(raw_sample)

    print("Sample file:", sample_file.name)
    print("RAW EXCERPT:")
    print(raw_sample[:800])
    print("" + "-" * 80 + "")
    print("CLEANED EXCERPT:")
    print(cleaned_sample["display_text"][:800])
else:
    print("No .txt files found. Check RAW_TRANSCRIPT_DIR.")

Found 17 transcript files
Sample file: #MLK _ But, If Not.txt
RAW EXCERPT:
But, if not, there was a day when many of the Israelites found themselves in bondage  in Babylon. There was a king of Babylon by the name of Nebuchadnezzar. You read about him  a good deal in the Book of Daniel and it stands as an epic that will remain  stencil on the mental sheets of unfolding generations. Nebuchadnezzar was a  mighty king and when he ruled he ruled and when he issued an order he meant business.  Nebuchadnezzar issued an order. He made a golden image and his order was that  everybody under the reign of his kingship had to bow before that golden image  and worship it. Now those of you who read the Bible remember that story. One day  Nebuchadnezzar called in the judges and the governors and the sheriffs and they  had a dedicatory service for this golden image and then he sa
--------------------------------------------------------------------------------
CLEANED EXCERPT:
But, if not, there was a d

## Process all transcripts

For each raw transcript, this creates three useful output files:

- `{name}_cleaned.txt` — cleaned text for reading and NLP
- `{name}_sentences.txt` — one sentence per line
- `{name}_analysis_lowercase.txt` — lowercase version for word frequency/token analysis


In [ ]:
summary_rows = []

for file_path in tqdm(txt_files, desc="Cleaning transcripts"):
    raw_text = file_path.read_text(encoding="utf-8")
    cleaned = clean_transcript(raw_text, lowercase_for_analysis=True)

    stem = file_path.stem

    cleaned_path = CLEANED_TRANSCRIPT_DIR / f"{stem}_cleaned.txt"
    sentence_path = CLEANED_TRANSCRIPT_DIR / f"{stem}_sentences.txt"
    analysis_path = CLEANED_TRANSCRIPT_DIR / f"{stem}_analysis_lowercase.txt"

    cleaned_path.write_text(cleaned["display_text"], encoding="utf-8")
    sentence_path.write_text(cleaned["sentence_text"], encoding="utf-8")
    analysis_path.write_text(cleaned["analysis_text"], encoding="utf-8")

    raw_word_count = len(raw_text.split())
    cleaned_word_count = len(cleaned["display_text"].split())
    sentence_count = len(cleaned["sentences"])


Cleaning transcripts:   0%|          | 0/17 [00:00<?, ?it/s]